# 제조 음성 ASR 심사 파이프라인

하나의 노트북에서 두 가지 데이터 모드를 사용합니다.

- `PUBLIC_PROXY`: 공개 Zeroth 한국어 음성으로 데이터 준비부터 모델 비교, 자동 프록시 선택, LoRA, 양자화, 고정 Test, 보고서·백데이터까지 전체 동작을 검증합니다.
- `PRIVATE_MANUFACTURING`: 나중에 승인된 제조 녹음과 검수 전사로 같은 코드를 다시 실행합니다. 이 모드의 모델·양자화 선택은 반드시 사람이 수행합니다.

> 공개 프록시 결과는 코드와 산출물의 정상 동작 증거입니다. 제조 현장 성능, 배포 적합성 또는 심사 최종 결론의 증거로 사용하면 안 됩니다.

**보안:** 카메라·마이크·패스키를 사용하지 않습니다. 실제 음성은 GitHub에 올리지 않고 승인된 비공개 Drive 경로만 사용합니다.

## 0. 실행 모드

In [ ]:
# "PUBLIC_PROXY" 또는 "PRIVATE_MANUFACTURING"
DATA_MODE = "PUBLIC_PROXY"

GITHUB_REPO_URL = "https://github.com/Pronesis9758/aias-specialist-asr.git"
GITHUB_BRANCH = "codex/whisper-benchmark-quantization"  # PR 병합 후 main
PROJECT_DIR = "/content/AIAS"
DRIVE_ROOT = "/content/drive/MyDrive/AI_Specialist_ASR_Project"

MODE_SETTINGS = {
    "PUBLIC_PROXY": {
        "config": "configs/public_proxy_assessment.yaml",
        "matrix": "configs/benchmarks/public_proxy_whisper_models.yaml",
        "quantization": "configs/quantization/public_proxy_whisper_quantization.yaml",
        "benchmark_id": "public-proxy-whisper-model-benchmark-v1",
        "quantization_id": "public-proxy-whisper-quantization-v1",
    },
    "PRIVATE_MANUFACTURING": {
        "config": "configs/manufacturing_private_template.yaml",
        "matrix": "configs/benchmarks/manufacturing_whisper_models_template.yaml",
        "quantization": "configs/quantization/manufacturing_whisper_quantization_template.yaml",
        "benchmark_id": "manufacturing-whisper-model-benchmark-v1",
        "quantization_id": "manufacturing-whisper-quantization-v1",
    },
}
if DATA_MODE not in MODE_SETTINGS:
    raise ValueError(f"지원하지 않는 DATA_MODE: {DATA_MODE}")

mode = MODE_SETTINGS[DATA_MODE]
CONFIG = mode["config"]
MODEL_MATRIX = mode["matrix"]
QUANTIZATION_SPEC = mode["quantization"]
BENCHMARK_ID = mode["benchmark_id"]
QUANTIZATION_ID = mode["quantization_id"]
PRIVATE_ROOT = f"{DRIVE_ROOT}/data/private/manufacturing"
IS_PUBLIC_PROXY = DATA_MODE == "PUBLIC_PROXY"
print("Mode:", DATA_MODE)
print("Config:", CONFIG)

## 1. GPU와 Google Drive 연결

In [ ]:
!nvidia-smi
from google.colab import drive

drive.mount("/content/drive")

## 2. GitHub 코드 동기화

In [ ]:
import os
import subprocess

if not os.path.exists(PROJECT_DIR):
    subprocess.run(
        [
            "git",
            "clone",
            "--branch",
            GITHUB_BRANCH,
            "--single-branch",
            GITHUB_REPO_URL,
            PROJECT_DIR,
        ],
        check=True,
    )
else:
    subprocess.run(
        ["git", "-C", PROJECT_DIR, "fetch", "origin", GITHUB_BRANCH],
        check=True,
    )
    subprocess.run(
        ["git", "-C", PROJECT_DIR, "checkout", GITHUB_BRANCH],
        check=True,
    )
    subprocess.run(
        [
            "git",
            "-C",
            PROJECT_DIR,
            "merge",
            "--ff-only",
            f"origin/{GITHUB_BRANCH}",
        ],
        check=True,
    )
os.chdir(PROJECT_DIR)
print("Git commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

## 3. 의존성 설치와 실행 함수

In [ ]:
%pip uninstall -y torchao
%pip install -q -e ".[train]" "transformers>=4.46,<5" "peft>=0.14,<0.19"

import sys


def run_aias(*args):
    command = [sys.executable, "-m", "aias_specialist.cli", *args]
    print("\nRunning:", " ".join(command), flush=True)
    environment = {**os.environ, "PYTHONUNBUFFERED": "1"}
    subprocess.run(command, check=True, env=environment)

## 4. 데이터 준비

`PUBLIC_PROXY`에서는 고정 revision의 `kresnik/zeroth_korean` 일부만 스트리밍해 Drive에 저장합니다. `PRIVATE_MANUFACTURING`에서는 기존 파일을 덮어쓰지 않고 입력 양식을 준비합니다.

In [ ]:
from pathlib import Path
import json
import shutil
import pandas as pd

if IS_PUBLIC_PROXY:
    run_aias("prepare-hf-dataset", "--config", CONFIG)
    public_root = Path(DRIVE_ROOT) / "data/public/zeroth_korean"
    display(pd.read_csv(public_root / "manifest.csv").groupby("split").size())
    provenance_path = public_root / "dataset_provenance.json"
    if provenance_path.exists():
        display(json.loads(provenance_path.read_text(encoding="utf-8")))
    print("PUBLIC PROXY: 제조 성능 증거가 아닌 전체동작 검증 데이터입니다.")
else:
    private_root = Path(PRIVATE_ROOT)
    (private_root / "audio").mkdir(parents=True, exist_ok=True)
    templates = {
        Path("data/templates/manufacturing_manifest_template.csv"): private_root / "manifest.csv",
        Path("data/templates/data_approval_template.yaml"): private_root / "data_approval.yaml",
        Path("data/templates/human_review_signoff_template.yaml"): private_root
        / "human_review_signoff.yaml",
        Path("configs/assessment/acceptance_criteria_template.yaml"): private_root
        / "acceptance_criteria.yaml",
    }
    for source, destination in templates.items():
        if not destination.exists():
            shutil.copy2(source, destination)
            print("Created:", destination)
        else:
            print("Preserved existing:", destination)
    run_aias(
        "assessment-audit",
        "--config",
        CONFIG,
        "--output-dir",
        f"{DRIVE_ROOT}/reports/assessment_readiness/private_manufacturing",
    )

## 5. Whisper 모델 비교

공개 프록시는 빠른 전체동작 검증을 위해 `tiny`, `base`, `small`을 비교합니다. 실제 제조 모드는 6개 후보를 비교합니다. 모델과 변환본은 Drive 캐시에 재사용됩니다.

In [ ]:
run_aias("model-matrix-lock", "--matrix", MODEL_MATRIX)
run_aias("benchmark-models", "--matrix", MODEL_MATRIX)
benchmark_dir = Path(DRIVE_ROOT) / "artifacts/benchmarks" / BENCHMARK_ID
benchmark_table = pd.read_csv(benchmark_dir / "benchmark_comparison.csv")
display(benchmark_table)

## 6. 모델 선택

공개 프록시는 완료 후보 중 rank 1을 자동 선택하지만 사람 검토로 기록하지 않습니다. 실제 제조 모드에서는 아래 사람 검토 값을 직접 입력해야 다음 단계로 진행됩니다.

In [ ]:
def best_completed_member(frame):
    completed = frame.loc[frame["status"].eq("completed")].copy()
    if completed.empty:
        raise RuntimeError("완료된 후보가 없습니다. 위 오류를 먼저 확인하세요.")
    completed["rank"] = pd.to_numeric(completed["rank"], errors="coerce")
    return completed.sort_values(
        ["rank", "cer", "wer", "aggregate_real_time_factor"],
        na_position="last",
    ).iloc[0]


if IS_PUBLIC_PROXY:
    selected_row = best_completed_member(benchmark_table)
    SELECTED_MODEL = str(selected_row["member_id"])
    REVIEWER = "AUTOMATED_PUBLIC_PROXY"
    MODEL_REASON = (
        "공개 Zeroth 프록시 rank 1 자동 선택. 코드·산출물 smoke test 전용이며 "
        "제조 모델 선정 또는 사람 검토 증거가 아님."
    )
    extra_selection_args = ["--automated-proxy"]
else:
    SELECTED_MODEL = "small"  # 비교표를 보고 수정
    REVIEWER = "TO_BE_COMPLETED"
    MODEL_REASON = "TO_BE_COMPLETED: 정확도·속도·메모리·거버넌스 근거"
    if "TO_BE_COMPLETED" in REVIEWER or "TO_BE_COMPLETED" in MODEL_REASON:
        raise ValueError("제조 모드에서는 사람 검토자와 모델 선택 근거를 입력하세요.")
    extra_selection_args = []

run_aias(
    "select-model",
    "--benchmark-dir",
    str(benchmark_dir),
    "--model-id",
    SELECTED_MODEL,
    "--reviewer",
    REVIEWER,
    "--reason",
    MODEL_REASON,
    *extra_selection_args,
)
model_selection = benchmark_dir / "model_selection.yaml"
print("Selected model:", SELECTED_MODEL)
print(model_selection.read_text(encoding="utf-8"))

## 7. 선택 모델 LoRA

공개 프록시는 40개 train 샘플·30 step의 짧은 실행으로 학습 코드, checkpoint, Base/LoRA 비교 산출물을 검증합니다. 실제 제조 모드는 별도 설정의 500 step을 사용하며 데이터 규모에 맞춰 조정합니다.

In [ ]:
run_aias(
    "train-selected-whisper",
    "--selection",
    str(model_selection),
    "--config",
    CONFIG,
)

## 8. 양자화 비교

In [ ]:
run_aias(
    "quantization-sweep",
    "--spec",
    QUANTIZATION_SPEC,
    "--selection",
    str(model_selection),
)
quantization_dir = Path(DRIVE_ROOT) / "artifacts/quantization" / QUANTIZATION_ID
quantization_table = pd.read_csv(quantization_dir / "quantization_comparison.csv")
display(quantization_table)

## 9. 양자화 선택

공개 프록시는 종합 rank 1을 자동 선택합니다. 실제 제조 모드는 정확도 손실, RTF, GPU 메모리와 모델 용량을 사람이 함께 검토합니다.

In [ ]:
if IS_PUBLIC_PROXY:
    selected_quantization_row = best_completed_member(quantization_table)
    SELECTED_VARIANT = str(selected_quantization_row["member_id"])
    QUANTIZATION_REASON = (
        "공개 Zeroth 프록시 종합 rank 1 자동 선택. 양자화 코드·산출물 smoke test "
        "전용이며 실제 배포 결정 또는 사람 검토 증거가 아님."
    )
    extra_quantization_args = ["--automated-proxy"]
else:
    SELECTED_VARIANT = "float16"  # 비교표를 보고 수정
    QUANTIZATION_REASON = "TO_BE_COMPLETED: 정확도 손실·속도·메모리·모델 용량 근거"
    if "TO_BE_COMPLETED" in QUANTIZATION_REASON:
        raise ValueError("제조 모드에서는 양자화 선택 근거를 입력하세요.")
    extra_quantization_args = []

run_aias(
    "select-quantization",
    "--quantization-dir",
    str(quantization_dir),
    "--variant-id",
    SELECTED_VARIANT,
    "--reviewer",
    REVIEWER,
    "--reason",
    QUANTIZATION_REASON,
    *extra_quantization_args,
)
quantization_selection = quantization_dir / "quantization_selection.yaml"
print("Selected variant:", SELECTED_VARIANT)
print(quantization_selection.read_text(encoding="utf-8"))

## 10. 고정 Test 최종평가

모델·양자화 선택이 끝난 뒤에만 그동안 보지 않은 Test split을 한 번 평가합니다.

In [ ]:
run_aias(
    "finalize-evaluation",
    "--selection",
    str(quantization_selection),
    "--config",
    CONFIG,
)

## 11. 산출물·심사 준비도 확인

공개 프록시에서는 `not_ready`가 정상입니다. 공개 데이터, 자동 선택, 미완료 사람 서명은 제조 심사 증거를 대체하지 못합니다.

In [ ]:
readiness_dir = Path(DRIVE_ROOT) / "reports/assessment_readiness" / DATA_MODE.lower()
run_aias(
    "assessment-audit",
    "--config",
    CONFIG,
    "--output-dir",
    str(readiness_dir),
)
readiness = json.loads((readiness_dir / "assessment_readiness.json").read_text(encoding="utf-8"))
print("Assessment readiness:", readiness["overall_status"])
display(
    pd.DataFrame(readiness["checks"])[["criterion", "check_id", "status", "message", "evidence"]]
)

## 12. 생성 결과 위치 요약

In [ ]:
expected_outputs = {
    "benchmark_table": benchmark_dir / "benchmark_comparison.csv",
    "benchmark_report": benchmark_dir / "reports/benchmark_report.docx",
    "model_selection": benchmark_dir / "model_selection.yaml",
    "selected_training": benchmark_dir / "selected_training_result.json",
    "quantization_table": quantization_dir / "quantization_comparison.csv",
    "quantization_report": quantization_dir / "reports/quantization_report.docx",
    "quantization_selection": quantization_dir / "quantization_selection.yaml",
    "final_test": quantization_dir / "final_test_result.json",
    "readiness_json": readiness_dir / "assessment_readiness.json",
    "readiness_markdown": readiness_dir / "assessment_readiness.md",
    "experiment_database": Path(DRIVE_ROOT) / "backdata/experiments.sqlite3",
}
display(
    pd.DataFrame(
        [
            {"artifact": name, "exists": path.exists(), "path": str(path)}
            for name, path in expected_outputs.items()
        ]
    )
)

if IS_PUBLIC_PROXY:
    print(
        "다음 단계: DATA_MODE을 PRIVATE_MANUFACTURING으로 바꾸고 승인된 제조 "
        "녹음·정답 전사를 넣은 뒤 같은 순서를 다시 실행합니다."
    )
else:
    print(
        "최종 보고서와 오류 샘플을 사람이 검수하고 human_review_signoff.yaml을 "
        "완료한 뒤 assessment-audit --fail-on-blocker로 최종 확인하세요."
    )